## Popularity-Based (Regularized Item Mean) Recommender (Baseline B)

This notebook implements a popularity-based recommender using a regularized item mean baseline.

The model assigns each restaurant a popularity score based on its average rating in the training data, adjusted using regularization so that restaurants with very few ratings are not unfairly ranked too highly above restaurants with many reviews. Recommendations are generated by returning the highest-scoring restaurants that each user has not already interacted with in the training set.

In [18]:
import pandas as pd
import math

# loading data
train_df = pd.read_csv("train_reviews_santa_barbara.csv")
test_df = pd.read_csv("test_reviews_santa_barbara.csv")
restaurants_df = pd.read_csv("restaurants_santa_barbara.csv")

# preprocessing
# set of all restaurants
all_restaurants = set(restaurants_df["business_id"].unique())

# dictionary mapping each user to set of restaurants they reviewed in training data
train_restaurants = {}
train_restaurants_groups = train_df.groupby("user_id")["business_id"]
for user, items in train_restaurants_groups:
    train_restaurants[user] = set(items)

# dictionary mapping each user to singular restaurant in testing data
test_restaurants = {}
for _, row in test_df.iterrows():
    test_restaurants[row["user_id"]] = row["business_id"]

# average global rating among all restaurants
global_mean = train_df["stars"].mean()
# smoothing parameter (can be changed as needed)
smoothing_param = 10

# create df with number of ratings and average rating for each restaurant in training set
restaurant_stats = train_df.groupby("business_id")["stars"].agg(["count", "mean"]).reset_index()

# calculate regularized item mean score and add it to the df
restaurant_stats["score"] = (restaurant_stats["count"] * restaurant_stats["mean"] + smoothing_param * global_mean) / (restaurant_stats["count"] + smoothing_param)

# sort the restaurants by regularized item mean score
popularity_ranking = restaurant_stats.sort_values("score", ascending=False)["business_id"].tolist()

recommendations = {}
for user in test_restaurants:
    # get set of restaurants the user has visited before
    seen_restaurants = train_restaurants.get(user, set())
    unseen_ranked = []
    for restaurant in popularity_ranking:
        # only add restaurants the user has not visited before
        if restaurant not in seen_restaurants:
            unseen_ranked.append(restaurant)
    # recommend top 30 (need 30 to test Hit/NDCG @ 10, 20, 30
    recommendations[user] = unseen_ranked[:30]


## Evaluation Metrics: Hit@k and NDCG@k

**Hit@K**: Measures whether the true test restaurant appears in the top-K recommendations

**NDCG@K**: Measures the ranking quality by assigning higher scores when the true restaurant is ranked higher in the recommendation list

In [19]:
def hit_at_k(k):
    """
    Computes hit@k for the recommender

    For each user, this metric checks whether the true/held-out test restaurant
    appears in the top-K recommended restaurants. The hit rate for each restaurant
    is either 0 or 1 (0 if it does not appear in top-K recommendations and 1 if it
    does appear in top-K). This returns the average hit rate across all users

    Parameters:
    k: int
        The cutoff rank (10, 20, 30) for evaluating recommendations

    Returns:
    float
        The average hit@k score across all users, the fraction of users whose true test
        restaurant appears in the top-K recommendations
    """
    hits = []
    for user, true_restaurant in test_restaurants.items():
        recs = recommendations[user]
        if true_restaurant in recs[0:k]:
            hits.append(1)
        else:
            hits.append(0)
    score = sum(hits) / len(hits)
    return score

def ndcg_at_k(k):
    """
    Computes Normalized Discounted Cumulative Gain (NDCG@K).

    For each user, this metric measures how highly the true (held-out) test
    restaurant is ranked in the top-K recommendations. Higher ranks receive
    higher scores, with a logarithmic discount applied to lower ranks.

    Since each user has only one relevant item, the ideal DCG (IDCG) is 1,
    so NDCG = DCG / IDCG = DCG / 1.

    Parameters:
    k: int
        The cutoff rank (10, 20, 30) for evaluating recommendations

    Returns:
    float
        The average NDCG@K score across all users
    """
    ndcgs = []

    for user, true_restaurant in test_restaurants.items():
        recs = recommendations[user]
        top_k = recs[:k]

        # idcg is 1 since there is only one restaurant in the test set
        idcg = 1.0

        if true_restaurant in top_k:
            rank = top_k.index(true_restaurant) + 1
            dcg = 1 / math.log2(rank + 1)
            ndcgs.append(dcg / idcg)
        else:
            dcg = 0.0
            ndcgs.append(dcg / idcg)

    score = sum(ndcgs) / len(ndcgs)
    return score

## Reporting final evaluation metrics

Evaluating Hit@10, Hit@20, Hit@30 and NDCG@10, NDCG@20, NDCG@30

In [20]:
print("Popularity-Based (Regularized Item Mean) Recommender: Evaluation Metrics")

for k in [10, 20, 30]:
    print(f"Hit@{k}:", hit_at_k(k), f"\t\tNDCG@{k}:", ndcg_at_k(k))

Popularity-Based (Regularized Item Mean) Recommender: Evaluation Metrics
Hit@10: 0.014996875650906061 		NDCG@10: 0.007734755929778665
Hit@20: 0.042282857737971254 		NDCG@20: 0.014617950149500599
Hit@30: 0.05269735471776713 		NDCG@30: 0.01684783554475835
